# FLUX HF LoRAs

⚠️ **Remember to copy this notebook in your Drive and rename.**

🤗 This notebook uses [Hugging Face Diffusers](https://huggingface.co/docs/diffusers/en/index) to create pipelines for tasks such as image generation.

*Workflows for IAAC MaCDA GenAI  (Apr - Jun 2026) taught by [James McBennett](https://www.linkedin.com/in/mcbennett/) and [Aymeric Brouez](https://www.linkedin.com/in/aymeric-brouez/)*

*With special thanks to past faculty [Nono Martínez Alonso](https://youtube.com/NonoMartinezAlonso).*

### Mount Drive

In [ ]:
# Mount Drive
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

## Setup

In [ ]:
%cd /content
!rm -rf iaac_genai
!git clone https://github.com/jamesmcbennett/iaac_genai
%cd /content/iaac_genai/

In [ ]:
import sys
sys.path.append('/content/iaac_genai')

In [ ]:
!pip install -q -r requirements.txt --quiet > /dev/null 2>&1

In [ ]:
from config import Config
from utils import set_image_path, save_image, save_yml, save_svg, save_gif
from IPython.display import Image as IPythonImage
import torch

In [ ]:
!pip install -q --upgrade diffusers transformers accelerate huggingface_hub peft torchao

## Hugging Face Token

In [ ]:
# Sign up at Hugging Face and create a "Read" access token (not the default "Fine-Grained" token).
# Click the 🔑 "Secrets" icon in the left sidebar.
# Enable Notebook Access, Set the Name to "HF_TOKEN", Paste your token as the Value

from google.colab import userdata
hf_token = userdata.get("HF_TOKEN")

## Output Directory

In [ ]:
Config.OUTPUT_DIR = '/content/drive/MyDrive/iaac_genai/outputs'

## Load pipeline

Load a pipeline with Hugging Face Diffusers.

In [ ]:
# Create pipeline.
from diffusers import FluxPipeline
pipe = FluxPipeline.from_pretrained("black-forest-labs/FLUX.1-schnell", torch_dtype=torch.bfloat16, token=hf_token)
pipe.enable_model_cpu_offload()

## Load LoRAs from Huggingface

In [ ]:
# LoRAs from Huggingface in list format [link, filename, trigger (optional)], Comment list_num
# Feel free to add/edit others as needed in the same format
LORA = [
    ('strangerzonehf/Ghibli-Flux-Cartoon-LoRA', 'Ghibili-Cartoon-Art.safetensors', 'Ghibli Art –'), #0
    ('strangerzonehf/Flux-Super-Realism-LoRA', 'super-realism.safetensors', 'Super Realism'), #1
]

## Config

You can override parameters here.

In [ ]:
SELECT_LORA = 0
lora = LORA[SELECT_LORA]
TRIGGER = lora[2] if len(lora) > 2 and lora[2] else ''
print("Selected LoRA:", lora[0], "\nTrigger:", TRIGGER)

base_PROMPT = "a bustling Middle Eastern souk, merchants and traders in traditional dress, camels laden with goods, narrow alleyways draped with colourful textiles, spice stalls, warm golden afternoon light, dust haze"
Config.PROMPT = f'{TRIGGER}, {base_PROMPT}' if TRIGGER else base_PROMPT

Config.SEED = 7797676568
Config.STEPS = 28

Config.AUTHOR = 'James'
Config.ALGO_TYPE = 'HF LoRAs'
Config.ALGO_NAME = 'FLUX'

Config.check()

## Generate

In [ ]:
# Unload previous LoRA if changing from previously run LoRA
pipe.unload_lora_weights()

In [ ]:
# Load LoRA
pipe.load_lora_weights(LORA[SELECT_LORA][0], weight_name=LORA[SELECT_LORA][1])

### Generate
generator = torch.Generator(Config.TORCH_DEVICE).manual_seed(Config.SEED)
image = pipe(Config.PROMPT, num_inference_steps=Config.STEPS, generator=generator).images[0]
set_image_path()

display(image)

## Save

Save the pipeline and config metadata, generated image, and the parameters image.

In [ ]:
# Save image
save_image(image)

In [ ]:
# Save yml metadata
save_yml(pipe)

In [ ]:
# Save svg parameters image
save_svg({
    'SEED': Config.SEED,
    'STEPS': Config.STEPS,
    'Google Colab': '',
})

##Alternatively: LoRA Loop (Seed)

In [ ]:
Config.SEED = range(0,2)
SELECT_LORA = 1
TRIGGER = lora[2] if len(lora) > 2 and lora[2] else ''
base_PROMPT = "portrait of an elderly fisherman, deeply weathered skin, thousand yard stare, backlit by evening sea light, medium format photography."
Config.PROMPT = f'{TRIGGER}, {base_PROMPT}' if TRIGGER else base_PROMPT

frames = []

for seed in Config.SEED:
  pipe.unload_lora_weights()
  ### Choose LoRA from list above using number
  Config.SEED = seed
  print("LoRA Loaded =", LORA[SELECT_LORA])
  print("Seed =", seed)
  # Load LoRA
  pipe.load_lora_weights(LORA[SELECT_LORA][0], weight_name=LORA[SELECT_LORA][1])

  ### Generate
  generator = torch.Generator(Config.TORCH_DEVICE).manual_seed(Config.SEED)
  image = pipe(Config.PROMPT, num_inference_steps=Config.STEPS, generator=generator).images[0]

  ### Add each image to frames
  frames.append(image)

  ### Set file name, save, and display
  set_image_path()
  save_image(image)
  display(image)

##GIF

In [ ]:
Config.FPS = 3
gif_path = save_gif(frames)
IPythonImage(filename=gif_path)

## Disconnect

In [ ]:
from google.colab import runtime
runtime.unassign()